# 14.14 怎樣讓多種終止深度都能使用？


希望使用者可選一輪、三輪或五輪，就要讓訓練真的關心這些終止深度。資料仍然可以是同一問題和同一個正確答案；要改的是模型在哪一輪交出答案（交卷），以及哪裡被計分。

第一種配方是**抽終止深度**。每次拿到訓練題，從集合抽 R，展開 R 輪，僅在 `head(s_R)` 算交叉熵。若 R 抽自1、3、5，每個終止深度都會在部分更新中被直接監督。最佳化的是這些深度代價的加權平均；抽樣機率越高，它在長期期望目標裡的比重越高。

第二種配方是**逐輪監督**（deep supervision）。一次展開到最大輪數，在多個中間狀態各算代價，再加權成一個總代價，例如 `(L_1+L_2+L_3)/3`，L_r 是第 r 輪的代價。它用更多讀出與代價計算，鼓勵早期也能交卷。下面是兩種目標的寫法示意，數字是人工設定的代價，不是實驗結果：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import random

loss_by_depth = {1: 1.2, 2: 0.8, 3: 0.7, 4: 0.9, 5: 0.6}
rng = random.Random(42)
R = rng.choice([1, 3, 5])
sampled_loss = loss_by_depth[R]
deep_loss = sum(loss_by_depth[r] for r in range(1, 6)) / 5
print("本次終止深度", R, "末端代價", sampled_loss, "逐輪平均代價", round(deep_loss, 4))

`loss_by_depth` 是[字典](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)，鍵是輪數，值是那一輪的代價。`random.Random(42)` 建立一個自己的抽樣工具，種子42固定它的起點，所以每次執行都抽到同一個 R；`rng.choice([1, 3, 5])` 從清單裡隨機挑一項。`sum(loss_by_depth[r] for r in range(1, 6)) / 5` 把第1到5輪的代價加起來再除以5，寫法和14.11數參數時相同。這次抽到 R=5，末端代價就是第5輪的0.6；逐輪平均是 `(1.2+0.8+0.7+0.9+0.6)/5=0.84`。`round(deep_loss, 4)` 只是取到小數第四位，避免印出浮點捨入的尾數。

在真正的 PyTorch 訓練裡，代價必須是保留求導記錄的 tensor，不能像這段展示程式換成 Python 浮點數後再做反向傳播。這裡只讓讀者核對「哪些項進入目標」；第19章的 Looped 解題器用的是逐輪監督，見[19.15](https://birdhackor.github.io/tiny-perceptron-vlm/19.15.html)。

同一個正確答案不奇怪：老師要求不同時間交卷都答同一題，並沒有要求內部狀態完全相同。早期輪數可能只能處理簡單題，後期可再處理複雜題；這是值得測試的學習結果，並不是代價寫下去就會出現。多個深度也會競爭共享參數，過度偏重一輪可能犧牲另一輪。

輪數多也會增加每次更新的工作，不能只把「更新步數相同」當成相同訓練成本。若做效果比較，記抽樣分布、平均展開輪數、監督位置、有效題數與實際時間；若只教機制，短配方足夠，不宣稱已重現研究規模效益。

練習：若抽到 R=5 的那一題，除第5輪外，又在第2、4輪各算一次交叉熵，還能叫「只在奇數終止深度監督」嗎？先回答再往下讀：不能，第2、4輪也被直接計分了。這正是下一節要守住的實驗界線。
